# Fastest Reliable Text → JSON Enrichment with Qwen3-8B-AWQ + vLLM

This notebook is built for Kaggle with your local Qwen3 AWQ model:

`/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1`

Production design:

- **vLLM offline inference** for throughput
- Local **Qwen3-8B-AWQ** model
- Deterministic JSON generation
- Optional vLLM structured outputs
- JSON parse + schema validation
- Retry only invalid outputs
- Resume-safe checkpointing
- Kaggle 2×T4 fixes: `disable_custom_all_reduce=True` and `VLLM_ATTENTION_BACKEND=TRITON_ATTN`

Important: on Kaggle 2×T4, vLLM tensor parallel startup may crash in custom all-reduce. This notebook disables custom all-reduce by default, forces Triton attention to avoid FlashInfer JIT/linker failures on Kaggle, and can fall back to single-GPU vLLM if TP=2 still fails.


In [1]:

# Cell 1 — Install runtime
# Run once. If Kaggle asks to restart after install, restart and continue from Cell 2.
# vLLM is used for high-throughput offline inference.
!pip -q install -U --no-cache-dir vllm transformers accelerate safetensors tqdm


## Kaggle/T4 vLLM startup note

Run this notebook after a fresh Kaggle **Restart Session** if a previous vLLM startup failed. Failed vLLM worker processes can leave GPU memory allocated, and vLLM will refuse to start if the requested `gpu_memory_utilization` exceeds currently free VRAM.

This version uses conservative T4-safe defaults:

- lower `gpu_memory_utilization`
- lower `max_num_seqs`
- `disable_custom_all_reduce=True`
- `enforce_eager=True`
- explicit `attention_backend='TRITON_ATTN'` when supported by the installed vLLM

After a successful 500-card test, you can tune upward from Cell 11.


In [2]:

# Cell 2 — Imports and environment

from __future__ import annotations

import gc
import json
import os
import re
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Iterable, Iterator

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7")
os.environ.setdefault("OMP_NUM_THREADS", "1")

# Kaggle/T4 reliability fix:
# vLLM may auto-select FLASHINFER, then FlashInfer JIT can fail on Kaggle with
# `/usr/bin/ld: cannot find -lcuda`. Force Triton attention before importing vLLM.
os.environ.setdefault("VLLM_ATTENTION_BACKEND", "TRITON_ATTN")

# Use spawn explicitly; vLLM would force this after CUDA init anyway.
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer

print("Python OK")
print("VLLM_ATTENTION_BACKEND:", os.environ.get("VLLM_ATTENTION_BACKEND"))
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} VRAM={p.total_memory/1024**3:.1f} GB")


Python OK
VLLM_ATTENTION_BACKEND: TRITON_ATTN
Torch: 2.11.0+cu130
CUDA: True
GPU 0: Tesla T4 VRAM=14.6 GB
GPU 1: Tesla T4 VRAM=14.6 GB


In [ ]:
# Cell 3 — Configuration

IS_KAGGLE = Path("/kaggle/working").exists()

# Full 2.4M v4 corpus.
# Upload court_authority_cards_v4.jsonl as a Kaggle dataset with this slug,
# or update DATASET_SLUG to match your actual dataset name.
DATASET_SLUG = "swiss-court-authority-cards-v4"
INPUT_FILENAME = "court_authority_cards_v4.jsonl"

KAGGLE_MODEL_PATH = Path("/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1")
HF_FALLBACK_MODEL_ID = "Qwen/Qwen3-8B-AWQ"

# Known line count for the full v4 artifact — avoids a full counting pass.
TOTAL_V4_LINES = 2_476_315


def first_existing_input_file() -> Path:
    candidates = [
        Path(f"/kaggle/input/{DATASET_SLUG}/{INPUT_FILENAME}"),
        Path(f"/kaggle/input/datasets/samiulislam180041221/{DATASET_SLUG}/{INPUT_FILENAME}"),
        Path("/kaggle/input/datasets") / INPUT_FILENAME,
        Path("/kaggle/input") / INPUT_FILENAME,
        Path("./") / INPUT_FILENAME,
    ]
    for p in candidates:
        if p.exists():
            return p

    if Path("/kaggle/input").exists():
        found = sorted(Path("/kaggle/input").glob(f"**/{INPUT_FILENAME}"))
        if found:
            return found[0]

    return candidates[0]


def first_existing_model_source() -> str:
    if KAGGLE_MODEL_PATH.exists():
        return str(KAGGLE_MODEL_PATH)
    return HF_FALLBACK_MODEL_ID


@dataclass
class Config:
    # Files
    input_file: Path = first_existing_input_file()
    output_file: Path = Path("/kaggle/working/court_authority_cards_v4_enriched.jsonl")
    failed_file: Path = Path("/kaggle/working/court_authority_cards_v4_enriched_failed.jsonl")
    checkpoint_file: Path = Path("/kaggle/working/court_authority_cards_v4_enriched_checkpoint.txt")

    # Model
    model_id: str = first_existing_model_source()
    quantization: str = "awq"
    dtype: str = "float16"
    max_model_len: int = 3072
    gpu_memory_utilization: float = 0.62

    # Tensor parallelism
    # Start with 2 on Kaggle 2xT4. Falls back to 1 if startup fails.
    tensor_parallel_size: int = 2

    # Kaggle 2xT4 reliability fixes.
    disable_custom_all_reduce: bool = True
    attention_backend: str = "TRITON_ATTN"
    enforce_eager: bool = True
    auto_shrink_memory_request: bool = True
    auto_fallback_to_single_gpu: bool = False

    # Throughput controls
    batch_size: int = 64
    max_num_seqs: int = 64

    # Prompt/output controls
    text_chars: int = 1400
    max_new_tokens: int = 224
    retry_max_new_tokens: int = 384

    # Accuracy/reliability controls
    temperature: float = 0.0
    top_p: float = 1.0
    repetition_penalty: float = 1.03
    main_enable_thinking: bool = False
    retry_enable_thinking: bool = True
    retry_invalid: bool = True
    use_structured_outputs: bool = True

    # Run controls
    # limit=500 for quality/throughput test; limit=0 for full 2.4M run.
    # At 5–10 cards/s on 2xT4, 2,476,315 cards ≈ 70–130 h (9–15 Kaggle sessions).
    # Resume is automatic: each new session reads from checkpoint_file.
    limit: int = 500
    start: int | None = None
    reset_output: bool = False
    flush_every_batches: int = 1

CONFIG = Config()

print("Input     :", CONFIG.input_file)
print("Output    :", CONFIG.output_file)
print("Failed    :", CONFIG.failed_file)
print("Checkpoint:", CONFIG.checkpoint_file)
print("Model     :", CONFIG.model_id)
print("TP size   :", CONFIG.tensor_parallel_size)
print("Disable custom all-reduce:", CONFIG.disable_custom_all_reduce)
print("Attention backend:", CONFIG.attention_backend)
print("Enforce eager:", CONFIG.enforce_eager)
print("Auto shrink memory request:", CONFIG.auto_shrink_memory_request)
print("Auto fallback TP=1:", CONFIG.auto_fallback_to_single_gpu)
print("Limit     :", CONFIG.limit)
print(f"Total v4 lines (known): {TOTAL_V4_LINES:,}")

In [4]:

# Cell 4 — Smoke test

def count_lines(path: Path) -> int:
    n = 0
    with path.open("rb") as f:
        for _ in f:
            n += 1
    return n


print("=" * 72)
print("Smoke Test")
print("=" * 72)

assert CONFIG.input_file.exists(), f"Input file not found: {CONFIG.input_file}"
print("Input exists:", CONFIG.input_file)
print("Input size GB:", CONFIG.input_file.stat().st_size / 1024**3)
print("Input lines:", count_lines(CONFIG.input_file))

with CONFIG.input_file.open("r", encoding="utf-8") as f:
    first = json.loads(next(f))
print("First citation:", first.get("citation"))
print("Fields:", sorted(first.keys()))

if Path(CONFIG.model_id).exists():
    model_path = Path(CONFIG.model_id)
    print("Local model path exists:", model_path)
    for name in ["config.json", "tokenizer_config.json"]:
        print(name, "OK" if (model_path / name).exists() else "MISSING")
    shards = list(model_path.glob("*.safetensors"))
    print("Safetensors shards:", len(shards))
else:
    print("Model is HF ID:", CONFIG.model_id)

tok = AutoTokenizer.from_pretrained(CONFIG.model_id, trust_remote_code=True)
probe = tok("Swiss Federal Tribunal").input_ids
print("Tokenizer OK. Probe tokens:", len(probe))

msg = [
    {"role": "system", "content": "Return JSON only."},
    {"role": "user", "content": "Test."},
]
rendered = tok.apply_chat_template(
    msg,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
print("Chat template OK. Rendered chars:", len(rendered))

CONFIG.output_file.parent.mkdir(parents=True, exist_ok=True)
test_write = CONFIG.output_file.parent / "_write_test.tmp"
test_write.write_text("ok", encoding="utf-8")
test_write.unlink()
print("Working dir writable")

print("=" * 72)
print("Smoke test passed")
print("=" * 72)


Smoke Test
Input exists: /kaggle/input/datasets/samiulislam180041221/swiss-court-authority-cards-rag-targets/court_authority_cards_v4_target_cards.jsonl
Input size GB: 0.9582597967237234
Input lines: 363258
First citation: BGE 139 I 2 E. 5.7
Fields: ['_enrichment_target', 'authority_role', 'citation', 'court_base', 'court_cases_cited', 'family', 'is_notification_paragraph', 'issue_labels_en', 'language', 'law_codes', 'legal_area', 'matched_terms_multilingual', 'pattern', 'provenance', 'retrieval_text_en', 'statutes_cited', 'structural', 'subfamily', 'summary_en_proxy', 'text_excerpt_original']
Local model path exists: /kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1
config.json OK
tokenizer_config.json OK
Safetensors shards: 2
Tokenizer OK. Probe tokens: 4
Chat template OK. Rendered chars: 121
Working dir writable
Smoke test passed


In [5]:

# Cell 5 — Schema, prompt, and validators

RAG_SCHEMA_HINT = {
    "english_summary": "string",
    "legal_topic": "string",
    "legal_question": "string",
    "legal_rule": "string",
    "court_holding": "string",
    "factual_context": "string",
    "english_legal_concepts": ["string"],
    "search_keywords": ["string"],
    "natural_language_queries": ["string"],
    "paragraph_role": "holding|reasoning|background|cost|procedural|disposition|standard_of_review|obiter",
    "outcome_signal": "granted|dismissed|inadmissible|remitted|partial|none",
}

REQUIRED_FIELDS = list(RAG_SCHEMA_HINT.keys())

ROLE_VALUES = {
    "holding",
    "reasoning",
    "background",
    "cost",
    "procedural",
    "disposition",
    "standard_of_review",
    "obiter",
}

OUTCOME_VALUES = {
    "granted",
    "dismissed",
    "inadmissible",
    "remitted",
    "partial",
    "none",
}

JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "english_summary": {"type": "string"},
        "legal_topic": {"type": "string"},
        "legal_question": {"type": "string"},
        "legal_rule": {"type": "string"},
        "court_holding": {"type": "string"},
        "factual_context": {"type": "string"},
        "english_legal_concepts": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": 8,
        },
        "search_keywords": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": 12,
        },
        "natural_language_queries": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": 5,
        },
        "paragraph_role": {"type": "string", "enum": sorted(ROLE_VALUES)},
        "outcome_signal": {"type": "string", "enum": sorted(OUTCOME_VALUES)},
    },
    "required": REQUIRED_FIELDS,
    "additionalProperties": False,
}

SYSTEM_PROMPT = (
    "You are a deterministic Swiss legal text-to-JSON extraction engine. "
    "The input is one paragraph from a Swiss court decision in German, French, or Italian, "
    "plus deterministic metadata. Return exactly one valid JSON object matching the requested schema. "
    "Use concise English legal terminology for semantic-search RAG. "
    "Use only statutes, articles, laws, and case citations present in the paragraph or metadata. "
    "Do not invent article numbers, laws, statutes, citations, facts, holdings, or outcomes. "
    "If a field is not supported, use an empty string, empty array, or 'none'. "
    "No markdown. No commentary. No explanations. No chain-of-thought."
)

REPAIR_GUARD = (
    "\n\nStrict repair instruction: return one valid compact JSON object only. "
    "No markdown fences, no prose, no explanation, no <think> block. "
    "All required keys must be present exactly once."
)


def safe_str(x: Any, max_chars: int = 1200) -> str:
    if x is None:
        return ""
    s = str(x)
    s = re.sub(r"\s+", " ", s).strip()
    return s[:max_chars]


def safe_list(x: Any, max_items: int = 8, max_chars: int = 120) -> list[str]:
    if x is None:
        return []
    if not isinstance(x, list):
        x = [x]
    out = []
    seen = set()
    for item in x:
        s = safe_str(item, max_chars=max_chars)
        if not s:
            continue
        key = s.lower()
        if key in seen:
            continue
        seen.add(key)
        out.append(s)
        if len(out) >= max_items:
            break
    return out


def build_user_prompt(card: dict[str, Any], *, text_chars: int) -> str:
    text = safe_str(card.get("text_excerpt_original", ""), max_chars=text_chars)

    metadata = {
        "citation": card.get("citation") or "",
        "language": card.get("language") or "",
        "court_base": card.get("court_base") or "",
        "authority_role": card.get("authority_role") or "",
        "legal_area": card.get("legal_area") or "",
        "family": card.get("family") or "",
        "subfamily": card.get("subfamily") or "",
        "issue_labels_en": card.get("issue_labels_en") or [],
        "law_codes": card.get("law_codes") or [],
        "statutes_cited": card.get("statutes_cited") or [],
        "court_cases_cited": card.get("court_cases_cited") or [],
        "summary_en_proxy": card.get("summary_en_proxy") or "",
        "retrieval_text_en": card.get("retrieval_text_en") or "",
        "structural": card.get("structural") or {},
    }

    return (
        "Create RAG-target JSON for this court-authority paragraph.\n\n"
        "Required JSON schema keys:\n"
        f"{json.dumps(RAG_SCHEMA_HINT, ensure_ascii=False)}\n\n"
        "Rules:\n"
        "- Return JSON object only.\n"
        "- Keep strings concise and legally precise.\n"
        "- The natural_language_queries must be useful English search queries for retrieval.\n"
        "- Use paragraph_role and outcome_signal only from the allowed enum values.\n"
        "- Do not invent facts or legal authorities.\n\n"
        "Metadata:\n"
        f"{json.dumps(metadata, ensure_ascii=False, sort_keys=True)}\n\n"
        "Original paragraph:\n"
        f"{text}\n"
    )


def strip_think_blocks(text: str) -> str:
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text.strip())
    return text.strip()


def extract_first_json_object(text: str) -> dict[str, Any]:
    text = strip_think_blocks(text)

    # Fast path.
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    # Balanced-object extraction.
    start = text.find("{")
    if start < 0:
        raise ValueError("no JSON object start found")

    depth = 0
    in_str = False
    esc = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    candidate = text[start : i + 1]
                    obj = json.loads(candidate)
                    if not isinstance(obj, dict):
                        raise ValueError("JSON root is not object")
                    return obj

    raise ValueError("no balanced JSON object found")


def normalize_and_validate(obj: dict[str, Any]) -> tuple[dict[str, Any], list[str]]:
    errors = []

    norm = {}
    for key in REQUIRED_FIELDS:
        if key not in obj:
            errors.append(f"missing:{key}")

    for key in [
        "english_summary",
        "legal_topic",
        "legal_question",
        "legal_rule",
        "court_holding",
        "factual_context",
    ]:
        norm[key] = safe_str(obj.get(key, ""), max_chars=1200)

    norm["english_legal_concepts"] = safe_list(obj.get("english_legal_concepts", []), max_items=8)
    norm["search_keywords"] = safe_list(obj.get("search_keywords", []), max_items=12)
    norm["natural_language_queries"] = safe_list(obj.get("natural_language_queries", []), max_items=5, max_chars=200)

    role = safe_str(obj.get("paragraph_role", "")).lower()
    if role not in ROLE_VALUES:
        errors.append(f"bad_paragraph_role:{role}")
        role = "reasoning"
    norm["paragraph_role"] = role

    outcome = safe_str(obj.get("outcome_signal", "")).lower()
    if outcome not in OUTCOME_VALUES:
        errors.append(f"bad_outcome_signal:{outcome}")
        outcome = "none"
    norm["outcome_signal"] = outcome

    # Minimum usefulness check.
    if not any(norm[k] for k in ["english_summary", "legal_question", "legal_rule", "court_holding"]):
        errors.append("empty_core_fields")

    if not norm["natural_language_queries"] and not norm["search_keywords"]:
        errors.append("empty_retrieval_terms")

    return norm, errors


def parse_validate_raw(raw: str) -> tuple[dict[str, Any] | None, list[str]]:
    try:
        obj = extract_first_json_object(raw)
    except Exception as exc:
        return None, [f"json_parse:{type(exc).__name__}:{str(exc)[:160]}"]

    norm, errors = normalize_and_validate(obj)
    if errors:
        return norm, errors
    return norm, []


In [6]:

# Cell 6 — vLLM generator

# vLLM imports. The attention backend is passed via LLM(..., attention_backend=...) when supported.
from vllm import LLM, SamplingParams

try:
    from vllm.sampling_params import StructuredOutputsParams
except Exception:
    StructuredOutputsParams = None


def gpu_free_fraction() -> float | None:
    """Return the minimum free/total VRAM fraction across visible CUDA devices."""
    if not torch.cuda.is_available():
        return None
    fractions = []
    for idx in range(torch.cuda.device_count()):
        try:
            free_b, total_b = torch.cuda.mem_get_info(idx)
            fractions.append(free_b / max(total_b, 1))
        except Exception:
            pass
    return min(fractions) if fractions else None


def shrink_memory_request_if_needed(config: Config, *, margin: float = 0.08) -> None:
    """Keep vLLM's requested memory below currently free VRAM.

    vLLM rejects startup when gpu_memory_utilization * total_vram > free_vram.
    This happens frequently in Kaggle after failed worker startups. A fresh restart is
    still best, but this prevents over-requesting memory.
    """
    if not getattr(config, "auto_shrink_memory_request", True):
        return
    frac = gpu_free_fraction()
    if frac is None:
        return
    safe = max(0.45, min(config.gpu_memory_utilization, frac - margin))
    if safe < config.gpu_memory_utilization:
        print(
            f"[vLLM] free VRAM fraction is about {frac:.2f}; "
            f"shrinking gpu_memory_utilization {config.gpu_memory_utilization:.2f} -> {safe:.2f}"
        )
        config.gpu_memory_utilization = safe


class VLLMJsonGenerator:
    def __init__(self, config: Config):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.model_id, trust_remote_code=True)

        if config.tensor_parallel_size > max(1, torch.cuda.device_count()):
            print(
                f"Requested TP={config.tensor_parallel_size}, but only {torch.cuda.device_count()} CUDA devices found. "
                "Using available device count."
            )
            config.tensor_parallel_size = max(1, torch.cuda.device_count())

        self.llm = self._load_llm_with_fallback(config)

        self.fast_params = self._make_params(
            max_tokens=config.max_new_tokens,
            structured=config.use_structured_outputs,
        )
        self.retry_params = self._make_params(
            max_tokens=config.retry_max_new_tokens,
            structured=config.use_structured_outputs,
        )

    def _llm_kwargs(self, config: Config) -> dict[str, Any]:
        kwargs = dict(
            model=config.model_id,
            quantization=config.quantization,
            dtype=config.dtype,
            trust_remote_code=True,
            tensor_parallel_size=config.tensor_parallel_size,
            gpu_memory_utilization=config.gpu_memory_utilization,
            max_model_len=config.max_model_len,
            max_num_seqs=config.max_num_seqs,
            enable_prefix_caching=True,
            disable_log_stats=True,
            enforce_eager=config.enforce_eager,

            # Critical Kaggle 2xT4 fix:
            # prevents vLLM custom_all_reduce.cuh invalid-argument startup failures.
            disable_custom_all_reduce=config.disable_custom_all_reduce,
        )
        # vLLM's current Python API supports attention_backend. Older builds may not;
        # _load_once() will retry without it if necessary.
        if getattr(config, "attention_backend", None):
            kwargs["attention_backend"] = config.attention_backend
        return kwargs

    def _load_once(self, config: Config) -> LLM:
        kwargs = self._llm_kwargs(config)
        try:
            return LLM(**kwargs)
        except TypeError as exc:
            # Some vLLM builds do not expose attention_backend on the Python LLM class.
            # Retry without it; the rest of the stability fixes still apply.
            if "attention_backend" in kwargs:
                print("[vLLM] attention_backend argument unsupported in this build; retrying without it.")
                kwargs.pop("attention_backend", None)
                return LLM(**kwargs)
            raise exc

    def _load_llm_with_fallback(self, config: Config) -> LLM:
        shrink_memory_request_if_needed(config)
        print("[vLLM] loading model:", config.model_id)
        print("[vLLM] quantization:", config.quantization)
        print("[vLLM] tensor_parallel_size:", config.tensor_parallel_size)
        print("[vLLM] gpu_memory_utilization:", config.gpu_memory_utilization)
        print("[vLLM] max_model_len:", config.max_model_len)
        print("[vLLM] max_num_seqs:", config.max_num_seqs)
        print("[vLLM] enforce_eager:", config.enforce_eager)
        print("[vLLM] disable_custom_all_reduce:", config.disable_custom_all_reduce)
        print("[vLLM] requested attention backend:", getattr(config, "attention_backend", None))

        try:
            return self._load_once(config)
        except Exception as exc:
            print("[vLLM] initial load failed:", repr(exc))
            print("[vLLM] This usually means stale worker processes or too little free VRAM.")
            print("[vLLM] Best fix: Kaggle Runtime -> Restart Session, then run with the safe defaults.")

            if not config.auto_fallback_to_single_gpu or config.tensor_parallel_size <= 1:
                raise

            print("[vLLM] falling back to tensor_parallel_size=1 with lower memory settings.")
            try:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except Exception:
                pass

            config.tensor_parallel_size = 1
            config.gpu_memory_utilization = min(config.gpu_memory_utilization, 0.55)
            config.max_num_seqs = min(config.max_num_seqs, 48)
            config.batch_size = min(config.batch_size, 48)
            config.max_model_len = min(config.max_model_len, 3072)
            shrink_memory_request_if_needed(config)
            return self._load_once(config)

    def _make_params(self, *, max_tokens: int, structured: bool) -> SamplingParams:
        kwargs = dict(
            temperature=self.config.temperature,
            top_p=self.config.top_p,
            max_tokens=max_tokens,
            repetition_penalty=self.config.repetition_penalty,
            stop=["<|im_end|>", "</s>"],
        )

        if structured and StructuredOutputsParams is not None:
            try:
                kwargs["structured_outputs"] = StructuredOutputsParams(json=JSON_SCHEMA)
                print("[vLLM] structured JSON outputs enabled")
            except Exception as exc:
                print("[vLLM] structured outputs unavailable, falling back to parser:", repr(exc))

        return SamplingParams(**kwargs)

    def render_prompt(self, card: dict[str, Any], *, enable_thinking: bool, extra_guard: str = "") -> str:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": build_user_prompt(card, text_chars=self.config.text_chars) + extra_guard,
            },
        ]

        try:
            return self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=enable_thinking,
            )
        except TypeError:
            # Older tokenizer fallback.
            return self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

    def generate_raw(
        self,
        cards: list[dict[str, Any]],
        *,
        retry: bool = False,
        extra_guard: str = "",
    ) -> list[str]:
        enable_thinking = self.config.retry_enable_thinking if retry else self.config.main_enable_thinking
        params = self.retry_params if retry else self.fast_params

        prompts = [
            self.render_prompt(card, enable_thinking=enable_thinking, extra_guard=extra_guard)
            for card in cards
        ]

        outputs = self.llm.generate(prompts, sampling_params=params, use_tqdm=False)
        return [out.outputs[0].text for out in outputs]


In [ ]:
# Cell 7 — IO, checkpointing, and main loop

def read_checkpoint(path: Path) -> int:
    if not path.exists():
        return 0
    try:
        return int(path.read_text(encoding="utf-8").strip() or "0")
    except Exception:
        return 0


def write_checkpoint(path: Path, next_line_idx: int) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(str(next_line_idx), encoding="utf-8")


def iter_cards(path: Path, *, start: int, stop_before: int | None) -> Iterator[tuple[int, dict[str, Any]]]:
    with path.open("r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if idx < start:
                continue
            if stop_before is not None and idx >= stop_before:
                break
            line = line.strip()
            if not line:
                continue
            try:
                card = json.loads(line)
            except Exception as exc:
                yield idx, {"_read_error": repr(exc), "_raw_line": line[:1000]}
                continue
            yield idx, card


def make_output_record(
    *,
    line_idx: int,
    card: dict[str, Any],
    rag_json: dict[str, Any] | None,
    raw: str,
    errors: list[str],
    source: str,
    elapsed_s: float,
) -> dict[str, Any]:
    rec = dict(card)
    rec["_source_line_idx"] = line_idx
    rec["_rag_generation"] = {
        "model": CONFIG.model_id,
        "engine": "vllm",
        "method": "qwen3_8b_vllm",
        "source": source,
        "elapsed_s": round(elapsed_s, 4),
        "errors": errors,
    }
    # rag_enrichment matches the schema consumed by experiment_enrichment_recall_stress.py
    # and all downstream BM25/vector retrieval pipelines.
    rec["rag_enrichment"] = rag_json
    if errors:
        rec["_raw_model_output"] = raw[:4000]
    return rec


def run_one_batch(
    generator: VLLMJsonGenerator,
    batch_pairs: list[tuple[int, dict[str, Any]]],
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], dict[str, int | float]]:
    batch_cards = [card for _, card in batch_pairs]
    t0 = time.time()

    raws = generator.generate_raw(batch_cards, retry=False)
    first_elapsed = time.time() - t0

    parsed = []
    retry_indices = []

    for j, raw in enumerate(raws):
        norm, errors = parse_validate_raw(raw)
        parsed.append((norm, errors, raw, "main"))
        if errors and CONFIG.retry_invalid:
            retry_indices.append(j)

    retry_elapsed = 0.0
    if retry_indices:
        retry_cards = [batch_cards[j] for j in retry_indices]
        t1 = time.time()
        retry_raws = generator.generate_raw(retry_cards, retry=True, extra_guard=REPAIR_GUARD)
        retry_elapsed = time.time() - t1

        for local_k, j in enumerate(retry_indices):
            raw2 = retry_raws[local_k]
            norm2, errors2 = parse_validate_raw(raw2)
            old_norm, old_errors, old_raw, old_source = parsed[j]
            if not errors2 or (old_norm is None and norm2 is not None):
                parsed[j] = (norm2, errors2, raw2, "retry")
            else:
                parsed[j] = (old_norm, old_errors, old_raw, old_source)

    ok_records = []
    failed_records = []

    total_elapsed = first_elapsed + retry_elapsed
    per_item_elapsed = total_elapsed / max(len(batch_pairs), 1)

    for (line_idx, card), (norm, errors, raw, source) in zip(batch_pairs, parsed):
        rec = make_output_record(
            line_idx=line_idx,
            card=card,
            rag_json=norm,
            raw=raw,
            errors=errors,
            source=source,
            elapsed_s=per_item_elapsed,
        )
        if errors:
            failed_records.append(rec)
        else:
            ok_records.append(rec)

    stats = {
        "batch_size": len(batch_pairs),
        "ok": len(ok_records),
        "failed": len(failed_records),
        "retried": len(retry_indices),
        "elapsed_s": total_elapsed,
        "cards_s": len(batch_pairs) / max(total_elapsed, 1e-9),
    }
    return ok_records, failed_records, stats


def append_jsonl(path: Path, records: list[dict[str, Any]]) -> None:
    if not records:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False, separators=(",", ":")) + "\n")
        f.flush()
        os.fsync(f.fileno())


def main(config: Config = CONFIG) -> None:
    total = TOTAL_V4_LINES  # use known count to skip a full counting pass

    if config.reset_output:
        for p in [config.output_file, config.failed_file, config.checkpoint_file]:
            if p.exists():
                p.unlink()

    start = config.start if config.start is not None else read_checkpoint(config.checkpoint_file)
    if start < 0:
        start = 0

    stop_before = None if config.limit == 0 else min(total, start + config.limit)

    print("Input      :", config.input_file)
    print("Output     :", config.output_file)
    print("Failures   :", config.failed_file)
    print("Checkpoint :", config.checkpoint_file)
    print("Model      :", config.model_id)
    print("Start      :", start)
    print(f"Total      : {total:,}")
    print("To process :", "all remaining" if stop_before is None else max(stop_before - start, 0))
    print("Batch size :", config.batch_size)
    print("Structured :", config.use_structured_outputs)
    print("Thinking   :", {"main": config.main_enable_thinking, "retry": config.retry_enable_thinking})

    generator = VLLMJsonGenerator(config)

    pending: list[tuple[int, dict[str, Any]]] = []
    processed = 0
    ok_total = 0
    fail_total = 0
    retry_total = 0
    t_run = time.time()
    last_line_idx = start

    total_for_bar = None if stop_before is None else max(stop_before - start, 0)
    pbar = tqdm(total=total_for_bar, desc="vllm-json", unit="card")

    def flush_batch():
        nonlocal pending, processed, ok_total, fail_total, retry_total, last_line_idx
        if not pending:
            return

        batch_pairs = pending
        pending = []

        ok_records, failed_records, stats = run_one_batch(generator, batch_pairs)
        append_jsonl(config.output_file, ok_records)
        append_jsonl(config.failed_file, failed_records)

        last_line_idx = batch_pairs[-1][0] + 1
        write_checkpoint(config.checkpoint_file, last_line_idx)

        processed += len(batch_pairs)
        ok_total += len(ok_records)
        fail_total += len(failed_records)
        retry_total += int(stats["retried"])

        if pbar:
            pbar.update(len(batch_pairs))

        elapsed = time.time() - t_run
        avg = processed / max(elapsed, 1e-9)
        remaining = max(total - last_line_idx, 0)
        eta_h = remaining / max(avg, 1e-9) / 3600
        eta_sessions = eta_h / 9.0

        print(
            f"[batch] size={stats['batch_size']} ok={stats['ok']} failed={stats['failed']} "
            f"retried={stats['retried']} batch_rate={stats['cards_s']:.2f} cards/s "
            f"avg={avg:.2f} cards/s eta_h={eta_h:.1f} sessions={eta_sessions:.1f} "
            f"checkpoint={last_line_idx:,}",
            flush=True,
        )

    try:
        for line_idx, card in iter_cards(config.input_file, start=start, stop_before=stop_before):
            pending.append((line_idx, card))
            if len(pending) >= config.batch_size:
                flush_batch()
        flush_batch()
    finally:
        if pbar:
            pbar.close()

    elapsed = time.time() - t_run
    avg = processed / max(elapsed, 1e-9)
    full_remaining = max(total - last_line_idx, 0)
    full_eta_h = full_remaining / max(avg, 1e-9) / 3600

    print("=" * 72)
    print("Done")
    print("Processed       :", processed)
    print("OK              :", ok_total)
    print("Failed          :", fail_total)
    print("Retried         :", retry_total)
    print(f"Elapsed min     : {elapsed / 60:.1f}")
    print(f"Average cards/s : {avg:.2f}")
    print(f"Checkpoint      : {read_checkpoint(config.checkpoint_file):,}")
    print(f"Full-run ETA h  : {full_eta_h:.1f}  ({full_eta_h / 9:.1f} Kaggle sessions)")
    print("=" * 72)

In [ ]:
# Cell 8 — Quality/throughput test (500 cards)
# Run this after a fresh Kaggle Restart Session.
# Inspect the cards/s in the log output, then use it to estimate full-run sessions:
#   total sessions ≈ 2,476,315 / (cards_s × 32,400 s/session)

CONFIG.limit = 500
CONFIG.batch_size = 64
CONFIG.max_num_seqs = 64
CONFIG.max_model_len = 3072
CONFIG.gpu_memory_utilization = 0.62
CONFIG.tensor_parallel_size = 2
CONFIG.disable_custom_all_reduce = True
CONFIG.enforce_eager = True
CONFIG.attention_backend = "TRITON_ATTN"
CONFIG.auto_shrink_memory_request = True
CONFIG.auto_fallback_to_single_gpu = False
CONFIG.reset_output = False

main(CONFIG)

In [ ]:
# Cell 9 — Inspect output quality

def read_jsonl_tail(path: Path, n: int = 3) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    lines = path.read_text(encoding="utf-8").splitlines()
    out = []
    for line in lines[-n:]:
        if line.strip():
            out.append(json.loads(line))
    return out


samples = read_jsonl_tail(CONFIG.output_file, 3)
for i, rec in enumerate(samples, 1):
    print("=" * 72)
    print("Sample", i)
    print("citation:", rec.get("citation"))
    print("generation:", rec.get("_rag_generation"))
    print(json.dumps(rec.get("rag_enrichment"), ensure_ascii=False, indent=2))

In [ ]:
# Cell 10 — Full 2.4M production run
# Run ONLY after Cell 9 quality check passes.
#
# Resume protocol (each new Kaggle session):
#   1. Restart Session
#   2. Re-run Cells 1–7 (installs, imports, config, schema, vLLM loader, IO functions)
#   3. Run this cell — it reads checkpoint_file and resumes automatically
#
# Output is appended, never overwritten. checkpoint_file persists in /kaggle/working.
# Download both output files and checkpoint_file before the session expires.

CONFIG.limit = 0           # process all remaining cards from checkpoint
CONFIG.reset_output = False  # never reset — always append and resume
main(CONFIG)

In [ ]:
# Cell 11 — Progress check and tuning presets

# --- Progress check (run any time) ---
def check_progress(config: Config = CONFIG) -> None:
    total = TOTAL_V4_LINES
    checkpoint = read_checkpoint(config.checkpoint_file)
    done_lines = 0
    if config.output_file.exists():
        with config.output_file.open("rb") as f:
            done_lines = sum(1 for _ in f)
    failed_lines = 0
    if config.failed_file.exists():
        with config.failed_file.open("rb") as f:
            failed_lines = sum(1 for _ in f)

    pct = 100.0 * checkpoint / max(total, 1)
    remaining = max(total - checkpoint, 0)
    print(f"Total input lines  : {total:,}")
    print(f"Checkpoint position: {checkpoint:,}  ({pct:.2f}%)")
    print(f"Output records     : {done_lines:,}")
    print(f"Failed records     : {failed_lines:,}")
    print(f"Remaining          : {remaining:,}")
    print()
    for cards_s in [3, 5, 8, 12]:
        eta_h = remaining / cards_s / 3600
        print(f"  ETA at {cards_s:2d} cards/s: {eta_h:.0f} h  ({eta_h / 9:.1f} Kaggle sessions)")

check_progress()

# --- Tuning presets ---

# SAFEST — if Kaggle reports low free VRAM or TP=2 startup fails:
# CONFIG.tensor_parallel_size = 1
# CONFIG.gpu_memory_utilization = 0.55
# CONFIG.batch_size = 48
# CONFIG.max_num_seqs = 48
# CONFIG.max_model_len = 3072
# CONFIG.enforce_eager = True

# FASTER — after a successful 500-card run, try these one step at a time:
# CONFIG.gpu_memory_utilization = 0.70
# CONFIG.batch_size = 96
# CONFIG.max_num_seqs = 96
# CONFIG.enforce_eager = False

# HIGHER ACCURACY — slower, better rag_enrichment quality:
# CONFIG.text_chars = 2000
# CONFIG.max_new_tokens = 320
# CONFIG.retry_max_new_tokens = 512
# CONFIG.retry_enable_thinking = True
# CONFIG.batch_size = 48
# CONFIG.max_num_seqs = 48

# LEAN — if memory is tight but want to maximise cards/s:
# CONFIG.text_chars = 1200
# CONFIG.max_new_tokens = 192
# CONFIG.retry_max_new_tokens = 320
# CONFIG.retry_enable_thinking = False
# CONFIG.batch_size = 96
# CONFIG.max_num_seqs = 96